# 01 - Download S&P 500 Data

Run this notebook weekly to refresh the current S&P 500 constituents and historical OHLCV data from Yahoo Finance. The saved parquet files are used by the ranking and portfolio target notebooks.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from clenow.data import (
    download_ohlcv,
    fetch_sp500_constituents,
    save_prices,
    with_extra_tickers,
)

In [2]:
# Weekly settings
HISTORY_PERIOD = "3y"
INTERVAL = "1d"
INDEX_PROXY = "SPY"
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

CONSTITUENTS_PATH = RAW_DIR / "sp500_constituents.parquet"
PRICES_PATH = PROCESSED_DIR / "sp500_prices.parquet"

In [3]:
constituents = fetch_sp500_constituents()
CONSTITUENTS_PATH.parent.mkdir(parents=True, exist_ok=True)
constituents.to_parquet(CONSTITUENTS_PATH, index=False)

print(f"Fetched {len(constituents)} S&P 500 constituents")
constituents.head()

Fetched 503 S&P 500 constituents


,ticker,yahoo_ticker,name,sector,industry
0,MMM,MMM,3M,Industrials,Industrial Conglomerates
1,AOS,AOS,A. O. Smith,Industrials,Building Products
2,ABT,ABT,Abbott Laboratories,Health Care,Health Care Equipment
3,ABBV,ABBV,AbbVie,Health Care,Biotechnology
4,ACN,ACN,Accenture,Information Technology,IT Consulting & Other Services


In [4]:
download_tickers = with_extra_tickers(constituents["yahoo_ticker"], [INDEX_PROXY])
prices = download_ohlcv(
    download_tickers,
    period=HISTORY_PERIOD,
    interval=INTERVAL,
    progress=True,
)
output_path = save_prices(prices, PRICES_PATH)

print(f"Saved {len(prices):,} rows to {output_path}")
print(f"Date range: {prices['date'].min().date()} to {prices['date'].max().date()}")
prices.head()

[*********************100%***********************]  504 of 504 completed


Saved 377,954 rows to /home/frederickpek/code/clenow/data/processed/sp500_prices.parquet
Date range: 2023-05-23 to 2026-05-22


Price,date,ticker,Open,High,Low,Close,Adj Close,Volume
0,2023-05-23,A,129.029999,130.539993,127.910004,128.639999,125.737244,2784400.0
1,2023-05-24,A,115.199997,121.089996,113.279999,120.989998,118.259842,6850400.0
2,2023-05-25,A,121.410004,121.410004,117.639999,119.489998,116.793709,3218900.0
3,2023-05-26,A,120.070000,120.720001,118.379997,120.419998,117.702721,2089100.0
4,2023-05-30,A,120.010002,121.309998,117.669998,117.730003,115.073433,2094800.0


In [5]:
coverage = (
    prices.groupby("ticker")
    .agg(first_date=("date", "min"), last_date=("date", "max"), rows=("date", "size"))
    .sort_values("rows")
)
coverage.head(20)

,first_date,last_date,rows
ticker,,,
Q,2025-10-27,2026-05-22,144
SNDK,2025-02-13,2026-05-22,320
GEV,2024-03-27,2026-05-22,541
SOLV,2024-03-26,2026-05-22,542
VLTO,2023-10-04,2026-05-22,661
FISV,2023-05-23,2026-05-22,752
WYNN,2023-05-23,2026-05-22,753
WY,2023-05-23,2026-05-22,753
WTW,2023-05-23,2026-05-22,753
